In [2]:
import os

# Check where the notebook thinks it is
print("Current working directory:", os.getcwd())


Current working directory: /Users/sunerawanni/Desktop/databricks


In [1]:
!pip install scikit-learn joblib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 4.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 3.5 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [2]:
import os
import joblib
import pandas as pd
from dotenv import load_dotenv
from databricks import sql
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

load_dotenv(dotenv_path="/Users/sunerawanni/Desktop/databricks/.env")
print("Ready ✅")

Ready ✅


In [14]:
host      = os.environ["DATABRICKS_HOST"].replace("https://", "")
http_path = os.environ["DATABRICKS_HTTP_PATH"]
token     = os.environ["DATABRICKS_TOKEN"]

with sql.connect(server_hostname=host, http_path=http_path, access_token=token) as conn:
    with conn.cursor() as cursor:
        cursor.execute("""
            SELECT item_id, category, confidence_score, confidence_tier, image_url
            FROM workspace.default.brand_detection_silver
        """)
        rows = cursor.fetchall()
        cols = [d[0] for d in cursor.description]

df = pd.DataFrame(rows, columns=cols)

# Re-encode labels
le_cat  = LabelEncoder()
le_tier = LabelEncoder()
df["category_enc"]        = le_cat.fit_transform(df["category"])
df["confidence_tier_enc"] = le_tier.fit_transform(df["confidence_tier"])

print(f"Loaded {len(df)} rows")
print(df.columns.tolist())
df.head()



Loaded 15702 rows
['item_id', 'category', 'confidence_score', 'confidence_tier', 'image_url', 'category_enc', 'confidence_tier_enc']


,item_id,category,confidence_score,confidence_tier,image_url,category_enc,confidence_tier_enc
0,851505458,ikat,0.3487,low,http://s3-eu-west-1.amazonaws.com/we-attribute...,6,1
1,851505459,plain,1.0000,high,http://s3-eu-west-1.amazonaws.com/we-attribute...,9,0
2,851505460,polka dot,0.6709,medium,http://s3-eu-west-1.amazonaws.com/we-attribute...,10,2
3,851505461,plain,1.0000,high,http://s3-eu-west-1.amazonaws.com/we-attribute...,9,0
4,851505462,geometry,0.7035,medium,http://s3-eu-west-1.amazonaws.com/we-attribute...,4,2


Feature Engineering

In [4]:
le_cat  = LabelEncoder()
le_tier = LabelEncoder()

df["category_enc"]        = le_cat.fit_transform(df["category"])
df["confidence_tier_enc"] = le_tier.fit_transform(df["confidence_tier"])

X = df[["confidence_score", "category_enc"]]
y = df["confidence_tier_enc"]

print("Classes:", le_tier.classes_)
print("Feature shape:", X.shape)


Classes: ['high' 'low' 'medium']
Feature shape: (15702, 2)


Train and evaluate

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=le_tier.classes_))


Accuracy: 0.9994
              precision    recall  f1-score   support

        high       1.00      1.00      1.00      2076
         low       1.00      1.00      1.00       328
      medium       1.00      1.00      1.00       737

    accuracy                           1.00      3141
   macro avg       1.00      1.00      1.00      3141
weighted avg       1.00      1.00      1.00      3141



In [6]:
os.makedirs("models", exist_ok=True)
joblib.dump(model,   "models/rf_model.pkl")
joblib.dump(le_cat,  "models/le_category.pkl")
joblib.dump(le_tier, "models/le_tier.pkl")
print("Model saved ✅")


Model saved ✅


Generate Predictions

In [7]:
# Generate predictions on full dataset
df["predicted_tier"]     = le_tier.inverse_transform(model.predict(X))
df["prediction_correct"] = df["predicted_tier"] == df["confidence_tier"]

print(f"Total predictions: {len(df)}")
print(f"Correct: {df['prediction_correct'].sum()}")
print(f"Overall accuracy: {df['prediction_correct'].mean():.4f}")
df[["item_id", "category", "confidence_score", "confidence_tier", "predicted_tier", "prediction_correct"]].head(10)

Total predictions: 15702
Correct: 15700
Overall accuracy: 0.9999


,item_id,category,confidence_score,confidence_tier,predicted_tier,prediction_correct
0,851505458,ikat,0.3487,low,low,True
1,851505459,plain,1.0000,high,high,True
2,851505460,polka dot,0.6709,medium,medium,True
3,851505461,plain,1.0000,high,high,True
4,851505462,geometry,0.7035,medium,medium,True
5,851505463,geometry,0.6585,medium,medium,True
6,851505464,plain,1.0000,high,high,True
7,851505465,plain,1.0000,high,high,True
8,851505466,floral,1.0000,high,high,True
9,851505467,plain,1.0000,high,high,True


Saving predictions back to Databricks


In [8]:
from datetime import datetime, timezone

rows_to_insert = []
for _, row in df.iterrows():
    rows_to_insert.append((
        int(row["item_id"]),
        str(row["category"]),
        float(row["confidence_score"]),
        str(row["confidence_tier"]),
        str(row["predicted_tier"]),
        bool(row["prediction_correct"]),
        datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    ))

with sql.connect(server_hostname=host, http_path=http_path, access_token=token) as conn:
    with conn.cursor() as cursor:

        # Create table
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS workspace.default.brand_detection_ml (
                item_id             BIGINT,
                category            STRING,
                confidence_score    DOUBLE,
                actual_tier         STRING,
                predicted_tier      STRING,
                prediction_correct  BOOLEAN,
                predicted_at        TIMESTAMP
            )
            USING DELTA
        """)

        # Clear old predictions
        cursor.execute("DELETE FROM workspace.default.brand_detection_ml")

        # Batch insert
        batch_size = 500
        for i in range(0, len(rows_to_insert), batch_size):
            batch = rows_to_insert[i:i+batch_size]
            values = ", ".join([f"({r[0]}, '{r[1]}', {r[2]}, '{r[3]}', '{r[4]}', {r[5]}, '{r[6]}')" for r in batch])
            cursor.execute(f"INSERT INTO workspace.default.brand_detection_ml VALUES {values}")
            print(f"Inserted batch {i//batch_size + 1}/{(len(rows_to_insert)//batch_size)+1}")

print("✅ Predictions saved to Databricks!")


Inserted batch 1/32
Inserted batch 2/32
Inserted batch 3/32
Inserted batch 4/32
Inserted batch 5/32
Inserted batch 6/32
Inserted batch 7/32
Inserted batch 8/32
Inserted batch 9/32
Inserted batch 10/32
Inserted batch 11/32
Inserted batch 12/32
Inserted batch 13/32
Inserted batch 14/32
Inserted batch 15/32
Inserted batch 16/32
Inserted batch 17/32
Inserted batch 18/32
Inserted batch 19/32
Inserted batch 20/32
Inserted batch 21/32
Inserted batch 22/32
Inserted batch 23/32
Inserted batch 24/32
Inserted batch 25/32
Inserted batch 26/32
Inserted batch 27/32
Inserted batch 28/32
Inserted batch 29/32
Inserted batch 30/32
Inserted batch 31/32
Inserted batch 32/32
✅ Predictions saved to Databricks!


verification with databricks

In [9]:
with sql.connect(server_hostname=host, http_path=http_path, access_token=token) as conn:
    with conn.cursor() as cursor:
        cursor.execute("""
            SELECT predicted_tier, COUNT(*) as count,
            ROUND(SUM(CASE WHEN prediction_correct THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS accuracy_pct
            FROM workspace.default.brand_detection_ml
            GROUP BY predicted_tier
            ORDER BY count DESC
        """)
        result = pd.DataFrame(cursor.fetchall(), columns=[d[0] for d in cursor.description])
print(result)


  predicted_tier  count accuracy_pct
0           high  10364       100.00
1         medium   3817        99.95
2            low   1521       100.00


Train a real classifier (predict category)


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# Features: confidence_score + tier encoded
# Target: category (16 classes)
X2 = df[["confidence_score", "confidence_tier_enc"]]
y2 = df["category_enc"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

model2 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
model2.fit(X2_train, y2_train)

y2_pred = model2.predict(X2_test)
print(f"Accuracy: {accuracy_score(y2_test, y2_pred):.4f}")
print("\nPer-class report:")
print(classification_report(y2_test, y2_pred, target_names=le_cat.classes_))


Accuracy: 0.4639

Per-class report:
              precision    recall  f1-score   support

      animal       0.07      0.03      0.04        71
     cartoon       0.06      0.15      0.09        52
     chevron       0.02      0.11      0.04        19
      floral       0.21      0.02      0.03       555
    geometry       0.09      0.14      0.11        69
 houndstooth       0.03      0.15      0.05        13
        ikat       0.09      0.08      0.09        71
 letter_numb       0.00      0.00      0.00        16
       other       0.17      0.17      0.17       103
       plain       0.64      0.83      0.72      1677
   polka dot       0.08      0.02      0.03       130
      scales       0.04      0.14      0.06        22
       skull       0.00      0.00      0.00         4
     squares       0.02      0.01      0.01        88
       stars       0.00      0.00      0.00         9
     stripes       0.12      0.01      0.03       140
      tribal       0.11      0.09      0.10  

In [11]:
#Feature importance
import pandas as pd

importances = pd.DataFrame({
    "feature": ["confidence_score", "confidence_tier_enc"],
    "importance": model2.feature_importances_
}).sort_values("importance", ascending=False)

print(importances)


               feature  importance
0     confidence_score    0.961367
1  confidence_tier_enc    0.038633


real model saved

In [12]:
joblib.dump(model2, "models/rf_category_model.pkl")
print("Category classifier saved ✅")

Category classifier saved ✅


In [15]:
# Extract domain/source patterns from image URLs as proxy features
df["url_length"]    = df["image_url"].str.len()
df["has_s3"]        = df["image_url"].str.contains("s3").astype(int)

# Add item_id-based features
df["id_mod_10"]     = df["item_id"] % 10
df["id_mod_100"]    = df["item_id"] % 100

X3 = df[["confidence_score", "confidence_tier_enc",
         "url_length", "has_s3", "id_mod_10", "id_mod_100"]]
y3 = df["category_enc"]

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42, stratify=y3
)

model3 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
model3.fit(X3_train, y3_train)

y3_pred = model3.predict(X3_test)
print(f"Accuracy: {accuracy_score(y3_test, y3_pred):.4f}")


Accuracy: 0.1452


In [19]:
!pip install tensorflow


  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached wheel-0.46.3-py3-none-any.whl.metadata (2.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 MB 5.6 MB/s  0:00:42m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 6.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 6.1 MB/s  0:00:00
Using cached wheel-0.46.3-py3-none-any.whl (30 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 5.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 6.2 MB/s  0:00:04m0:00:0100:01
Using cached setuptools-82.0.0-py3-none-any.whl (1.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [tensorflow]0 [tensorflow]


In [20]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, accuracy_score

# ── 1. Load dataset ──────────────────────────────────────────────────────────
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Class names
class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

# ── 2. Normalize ─────────────────────────────────────────────────────────────
X_train = X_train / 255.0
X_test  = X_test  / 255.0

# Reshape for CNN (add channel dimension)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# ── 3. Build CNN ─────────────────────────────────────────────────────────────
model_cnn = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

model_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_cnn.summary()


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Train: (60000, 28, 28) | Test: (10000, 28, 28)


/Users/sunerawanni/Desktop/databricks/.venv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
# ── 4. Train ─────────────────────────────────────────────────────────────────
history = model_cnn.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)


Epoch 1/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.8036 - loss: 0.5416 - val_accuracy: 0.8642 - val_loss: 0.3764
Epoch 2/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.8694 - loss: 0.3563 - val_accuracy: 0.8813 - val_loss: 0.3155
Epoch 3/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.8863 - loss: 0.3112 - val_accuracy: 0.8917 - val_loss: 0.2950
Epoch 4/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.8966 - loss: 0.2850 - val_accuracy: 0.8947 - val_loss: 0.2790
Epoch 5/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9064 - loss: 0.2581 - val_accuracy: 0.9070 - val_loss: 0.2581
Epoch 6/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9116 - loss: 0.2385 - val_accuracy: 0.9035 - val_loss: 0.2640
Epoch 7/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9177 - loss: 0.2224 - val_accuracy: 0.9133 - val_loss: 0.2423
Epoch 8/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.9214 - loss: 0.2084 - val_accuracy: 0.

In [22]:
# ── 5. Evaluate ──────────────────────────────────────────────────────────────
test_loss, test_acc = model_cnn.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc:.4f}")

y_pred = np.argmax(model_cnn.predict(X_test), axis=1)
print(classification_report(y_test, y_pred, target_names=class_names))



Test Accuracy: 0.9119
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

     T-shirt       0.87      0.85      0.86      1000
     Trouser       1.00      0.97      0.99      1000
    Pullover       0.85      0.89      0.87      1000
       Dress       0.88      0.93      0.90      1000
        Coat       0.85      0.86      0.86      1000
      Sandal       0.98      0.98      0.98      1000
       Shirt       0.78      0.73      0.75      1000
     Sneaker       0.95      0.97      0.96      1000
         Bag       0.98      0.98      0.98      1000
  Ankle boot       0.97      0.96      0.97      1000

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000



In [23]:
# ── 6. Save model ─────────────────────────────────────────────────────────────
os.makedirs("models", exist_ok=True)
model_cnn.save("models/fashion_cnn.keras")
print("CNN model saved ✅")


CNN model saved ✅


saving predictions back to databricks


In [24]:
# Build predictions dataframe
pred_labels = [class_names[p] for p in y_pred]
true_labels = [class_names[t] for t in y_test]

pred_df = pd.DataFrame({
    "item_index"   : range(len(y_test)),
    "true_label"   : true_labels,
    "predicted_label": pred_labels,
    "correct"      : [p == t for p, t in zip(pred_labels, true_labels)]
})

# Save to Databricks
from datetime import datetime, timezone
now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

with sql.connect(server_hostname=host, http_path=http_path, access_token=token) as conn:
    with conn.cursor() as cursor:
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS workspace.default.fashion_mnist_predictions (
                item_index      BIGINT,
                true_label      STRING,
                predicted_label STRING,
                correct         BOOLEAN,
                predicted_at    TIMESTAMP
            ) USING DELTA
        """)
        cursor.execute("DELETE FROM workspace.default.fashion_mnist_predictions")

        batch_size = 500
        rows = [(int(r.item_index), r.true_label, r.predicted_label,
                 bool(r.correct), now) for _, r in pred_df.iterrows()]

        for i in range(0, len(rows), batch_size):
            batch = rows[i:i+batch_size]
            values = ", ".join([
                f"({r[0]}, '{r[1]}', '{r[2]}', {r[3]}, '{r[4]}')"
                for r in batch
            ])
            cursor.execute(f"INSERT INTO workspace.default.fashion_mnist_predictions VALUES {values}")
        print(f"✅ {len(rows)} predictions saved to Databricks!")


✅ 10000 predictions saved to Databricks!
